# Importing and preparing supermarkets data

## Libraries and settings

In [1]:
# Libraries
import os
import fnmatch
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print('Current working directory:', os.getcwd())

# Show .json files in the current working directory
flist = fnmatch.filter(os.listdir('.'), '*.json')
for i in flist:
    print(i)

Current working directory: /workspaces/data_analytics/Week_02
supermarkets.json


## Importing data

In [2]:
# Read the data to a pandas data frame
df1 = pd.read_json('supermarkets.json', encoding='utf-8')
df1.head(5)

,type,id,lat,lon,tags
0,node,33126515,47.155616,9.037915,"{'brand': 'Spar', 'brand:wikidata': 'Q610492',..."
1,node,36726161,47.226191,8.980329,"{'addr:city': 'Uznach', 'addr:housenumber': '2..."
2,node,39768209,47.225069,8.969981,"{'addr:city': 'Uznach', 'addr:postcode': '8730..."
3,node,39947904,47.376732,8.542161,"{'addr:city': 'Zürich', 'addr:country': 'CH', ..."
4,node,48932835,47.375020,8.522895,"{'addr:city': 'Zürich', 'addr:housenumber': '7..."


## Count number of rows and columns in the data frame

In [3]:
# Dimension (rows, columns)
print('Dimension:', df1.shape)

# Number of rows
print('Number of rows:', df1.shape[0])

# Number of columns
print('Number of columns:', df1.shape[1])

Dimension: (3392, 5)
Number of rows: 3392
Number of columns: 5


## Column 'tags' is a pandas Series with dictionaries -> change to data frame

In [4]:
# Type of the first item of column tags
print(type(df1.tags))
print(type(df1.tags[0]))

# Content of the first item of column tags
print(df1.tags[0].keys())

# Change to data frame
df2 = pd.DataFrame.from_records(df1.tags)
df2 = df2[['brand', 'shop', 'addr:city', 'addr:street', 'addr:housenumber', 'addr:postcode']]

# Rename selected columns
df2 = df2.rename(columns={'addr:city': 'city',
                          'addr:street':'street',
                          'addr:housenumber': 'housenumber',
                          'addr:postcode': 'postcode'})

# Show first records of data frame
df2.head()

<class 'pandas.core.series.Series'>
<class 'dict'>
dict_keys(['brand', 'brand:wikidata', 'brand:wikipedia', 'name', 'opening_hours', 'shop'])


,brand,shop,city,street,housenumber,postcode
0,Spar,supermarket,NaN,NaN,NaN,NaN
1,Migros,supermarket,Uznach,Zürcherstrasse,25,8730
2,Coop,supermarket,Uznach,NaN,NaN,8730
3,Coop,supermarket,Zürich,Bahnhofbrücke,1,8001
4,Migros,supermarket,Zürich,Wengistrasse,7,8004


## Merge df1 and df2

In [5]:
# Merge df and df2
df = pd.merge(df1[['type', 'id', 'lat', 'lon']], 
              df2[['brand', 'shop', 'city', 'street', 'housenumber', 'postcode']],
              left_index=True, 
              right_index=True)
df.head(5)

,type,id,lat,lon,brand,shop,city,street,housenumber,postcode
0,node,33126515,47.155616,9.037915,Spar,supermarket,NaN,NaN,NaN,NaN
1,node,36726161,47.226191,8.980329,Migros,supermarket,Uznach,Zürcherstrasse,25,8730
2,node,39768209,47.225069,8.969981,Coop,supermarket,Uznach,NaN,NaN,8730
3,node,39947904,47.376732,8.542161,Coop,supermarket,Zürich,Bahnhofbrücke,1,8001
4,node,48932835,47.375020,8.522895,Migros,supermarket,Zürich,Wengistrasse,7,8004


## Count and identify the number of missing values (if any)

In [6]:
# Count missing values
print(pd.isna(df).sum())

# Identify rows with missing values, e.g.:
df.loc[pd.isna(df['city'])]

type              0
id                0
lat               0
lon               0
brand          1065
shop              0
city           1777
street         1608
housenumber    1680
postcode       1709
dtype: int64


,type,id,lat,lon,brand,shop,city,street,housenumber,postcode
0,node,33126515,47.155616,9.037915,Spar,supermarket,NaN,NaN,NaN,NaN
5,node,60271452,47.406671,9.305450,NaN,supermarket,NaN,NaN,NaN,NaN
6,node,70656485,47.491253,8.733981,NaN,supermarket,NaN,NaN,NaN,NaN
10,node,81321513,47.532917,9.066408,Landi,supermarket,NaN,NaN,NaN,NaN
13,node,95582038,47.050385,9.059214,NaN,supermarket,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
3384,node,11083317088,46.862184,9.531169,Lidl,supermarket,NaN,NaN,NaN,NaN
3386,node,11098091830,46.205111,6.130174,Coop,supermarket,NaN,NaN,NaN,NaN
3387,node,11099817248,46.928691,7.561873,NaN,supermarket,NaN,NaN,NaN,NaN
3388,node,11103235832,46.166742,8.771970,Migros,supermarket,NaN,NaN,NaN,NaN


## Count and identify duplicated values (if any)

In [7]:
# Count duplicated values
print(df.duplicated().sum())

# Identify rows with duplicated values, e.g.:
df[df[['id']].duplicated()]

0


,type,id,lat,lon,brand,shop,city,street,housenumber,postcode


## Get data types of all variables

In [8]:
# Get data types (note that in pandas, a string is referred to as 'object')
df.dtypes

type            object
id               int64
lat            float64
lon            float64
brand           object
shop            object
city            object
street          object
housenumber     object
postcode        object
dtype: object

## Additional filters on supermarkets

The `.loc` method lets us filter a data frame with boolean conditions, e.g.:
```python
df_filtered = df.loc[(df['brand'] == 'Coop') & (df['city'] == 'Zürich')]
```

In [9]:
# f) Filter only Migros supermarkets in the city of Zürich
df_migros_zuerich = df.loc[(df['brand'] == 'Migros') & (df['city'] == 'Zürich')]

print('Number of Migros supermarkets in Zürich:', df_migros_zuerich.shape[0])
df_migros_zuerich.head()

Number of Migros supermarkets in Zürich: 30


,type,id,lat,lon,brand,shop,city,street,housenumber,postcode
4,node,48932835,47.375020,8.522895,Migros,supermarket,Zürich,Wengistrasse,7,8004
11,node,83330862,47.344749,8.529981,Migros,supermarket,Zürich,Etzelstrasse,3,8038
16,node,119249170,47.375255,8.536107,Migros,supermarket,Zürich,Löwenstrasse,31-35,8001
50,node,262400822,47.364072,8.530945,Migros,supermarket,Zürich,Tessinerplatz,10,8002
71,node,267346993,47.385598,8.531471,Migros,supermarket,Zürich,Limmatstrasse,152,8005


In [10]:
# g) Filter and count all Coop supermarkets in Zürich, Basel & Bern
df_coop_zbb = df.loc[(df['brand'] == 'Coop') & (df['city'].isin(['Zürich', 'Basel', 'Bern']))]

print('Number of Coop supermarkets in Zürich, Basel & Bern:', df_coop_zbb.shape[0])
df_coop_zbb['city'].value_counts()

Number of Coop supermarkets in Zürich, Basel & Bern: 52


city
Zürich    37
Basel     10
Bern       5
Name: count, dtype: int64

In [11]:
# h) Filter supermarkets with available brand, city, housenumber and postcode
df_complete = df.loc[df['brand'].notna() &
                     df['city'].notna() &
                     df['housenumber'].notna() &
                     df['postcode'].notna()]

print('Number of supermarkets with complete brand/city/housenumber/postcode:', df_complete.shape[0])
df_complete.head()

Number of supermarkets with complete brand/city/housenumber/postcode: 1148


,type,id,lat,lon,brand,shop,city,street,housenumber,postcode
1,node,36726161,47.226191,8.980329,Migros,supermarket,Uznach,Zürcherstrasse,25,8730
3,node,39947904,47.376732,8.542161,Coop,supermarket,Zürich,Bahnhofbrücke,1,8001
4,node,48932835,47.375020,8.522895,Migros,supermarket,Zürich,Wengistrasse,7,8004
7,node,70656488,47.491874,8.706448,Migros,supermarket,Winterthur,Zürcherstrasse,102,8406
8,node,75749133,47.340967,8.530601,ALDI,supermarket,Zürich,Albisstrasse,81,8038


In [12]:
# i) Include opening hours as an additional variable in the data frame
# 'opening_hours' lives inside the same 'tags' dict as 'brand', 'shop', etc. (see df1.tags[0].keys() above),
# so we pull it out of df1.tags the same way df2 was built, then attach it to df by matching row position.
df['opening_hours'] = df1.tags.apply(lambda t: t.get('opening_hours'))

print('Missing opening_hours:', df['opening_hours'].isna().sum(), 'out of', df.shape[0])
df[['brand', 'city', 'opening_hours']].head()

Missing opening_hours: 1361 out of 3392


,brand,city,opening_hours
0,Spar,NaN,Mo-Th 08:00-19:00; Fr 08:00-20:00; Sa 08:00-17:00
1,Migros,Uznach,"Mo-Th 08:00-19:00, Fr 08:00-20:00, Sa 07:30-17..."
2,Coop,Uznach,None
3,Coop,Zürich,Mo-Sa 06:00-22:00
4,Migros,Zürich,Mo-Sa 08:00-21:00; PH off


In [13]:
# j) Filter supermarkets with available opening hours
df_opening_hours = df.loc[df['opening_hours'].notna()]

print('Number of supermarkets with known opening hours:', df_opening_hours.shape[0])
df_opening_hours[['brand', 'city', 'opening_hours']].head()

Number of supermarkets with known opening hours: 2031


,brand,city,opening_hours
0,Spar,NaN,Mo-Th 08:00-19:00; Fr 08:00-20:00; Sa 08:00-17:00
1,Migros,Uznach,"Mo-Th 08:00-19:00, Fr 08:00-20:00, Sa 07:30-17..."
3,Coop,Zürich,Mo-Sa 06:00-22:00
4,Migros,Zürich,Mo-Sa 08:00-21:00; PH off
7,Migros,Winterthur,Mo-Fr 07:30-20:00; PH off; Sa 08:00-19:00


### Save data to file

In [14]:
df.to_csv('supermarkets_data_prepared.csv', 
          sep=",", 
          encoding='utf-8',
          index=False)

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [15]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-09-24 10:53:44
Python Version: 3.11.16
-----------------------------------
